# SAL_W_WARD DEDUPLICATION

**Tess Vu**

This notebook goes over Jill's ArcGIS Pro spatial join output `sal_w_ward` to address duplicates.

In [ ]:
import geopandas as gpd
import pandas as pd
import os

In [ ]:
import sys
from pathlib import Path

ROOT = Path("..").resolve()          # repository root; notebooks run from notebooks/
DATA = ROOT / "data"
RAW = DATA / "raw"
INTERMEDIATE = DATA / "intermediate"
FINAL = DATA / "final"

IN_SAL_WARD_JOIN = RAW / "sal_w_ward_new" / "sal_w_ward_new.shp"
OUT_SAL_DEDUP = INTERMEDIATE / "sal_w_ward_dedup" / "sal_w_ward_dedup.shp"


In [ ]:
# LOAD THE ORIGINAL SHAPEFILE.
sal_raw = gpd.read_file(IN_SAL_WARD_JOIN)

print(f"ORIGINAL: {sal_raw.shape[0]} features")
print(f"Unique EA_CODEs: {sal_raw['EA_CODE'].nunique()}")
print(f"Duplicate EA_CODE rows: {sal_raw.shape[0] - sal_raw['EA_CODE'].nunique()}")
print()
print(f"AREA column range: {sal_raw['AREA'].min():.2f} to {sal_raw['AREA'].max():.2f}")
print(f"PERCENTAGE column range: {sal_raw['PERCENTAGE'].min():.4f} to {sal_raw['PERCENTAGE'].max():.4f}")

In [ ]:
# PRE-DEDUPLICATION CROSS-CHECK 1: DUPLICATE STRUCTURE.
# Isolate all rows whose EA_CODE appears more than once.
# Each duplicated EA_CODE should appear exactly twice (one SAL straddling
# two wards), but checking for cases with 3+ occurrences is important.
dup_counts = sal_raw.groupby("EA_CODE").size()
n_appear_once   = (dup_counts == 1).sum()
n_appear_twice  = (dup_counts == 2).sum()
n_appear_more   = (dup_counts > 2).sum()

print("DUPLICATE OCCURRENCE COUNTS")
print(f"  EA_CODEs appearing exactly once  : {n_appear_once}")
print(f"  EA_CODEs appearing exactly twice : {n_appear_twice}")
print(f"  EA_CODEs appearing 3 or more times: {n_appear_more}")
if n_appear_more > 0:
    print(f"  WARNING: EA_CODEs with 3+ rows (inspect these manually):")
    print(dup_counts[dup_counts > 2].head(10))
print()

In [ ]:
# PRE-DEDUPLICATION CROSS-CHECK 2: WARD ASSIGNMENTS DIFFER BETWEEN DUPLICATES.
# The working hypothesis is that duplicates arise from SALs straddling
# two wards, meaning each duplicate pair should have two distinct WardID
# values. If WardIDs are identical within a duplicate group, the duplicate
# has a different cause and needs separate handling.
dup_ea_codes = dup_counts[dup_counts > 1].index
dup_rows = sal_raw[sal_raw["EA_CODE"].isin(dup_ea_codes)].copy()

ward_col = "census_war"

if ward_col in dup_rows.columns:
    ward_per_dup = dup_rows.groupby("EA_CODE")[ward_col].nunique()
    same_ward = (ward_per_dup == 1).sum()
    diff_ward = (ward_per_dup > 1).sum()

    print("WARD ASSIGNMENT CHECK ACROSS DUPLICATE PAIRS")
    print(f"  Duplicate groups with different wards (expected): {diff_ward}")
    print(f"  Duplicate groups with same ward (unexpected):     {same_ward}")

    if same_ward > 0:
        same_ward_codes = ward_per_dup[ward_per_dup == 1].index[:5]
        print(f"  Sample same-ward duplicates (inspect these):")
        print(dup_rows[dup_rows["EA_CODE"].isin(same_ward_codes)][["EA_CODE", ward_col, "AREA", "PERCENTAGE"]])
else:
    print(f"WARNING: Ward column '{ward_col}' not found. Available columns:")
    print([c for c in sal_raw.columns if "ward" in c.lower() or "WAR" in c])
print()

In [ ]:
# DIAGNOSTIC: FULL INSPECTION OF EA_CODEs WITH 3 OR MORE OCCURRENCES.
# Check 1 flagged 22 such EA_CODEs. This cell prints all of them and
# shows every row, to confirm whether the extra copies are also
# identical duplicates or carry distinct attribute values.
high_dup_codes = dup_counts[dup_counts > 2].index
print(f"ALL EA_CODEs WITH 3+ OCCURRENCES: {len(high_dup_codes)} total")
print()

high_dup_rows = sal_raw[sal_raw["EA_CODE"].isin(high_dup_codes)].copy()

inspect_cols = ["EA_CODE", ward_col, "AREA", "PERCENTAGE"]

for ea_code, group in high_dup_rows.groupby("EA_CODE"):
    n = len(group)
    n_unique_wards = group[ward_col].nunique()
    n_unique_areas = group["AREA"].nunique()
    n_unique_pcts  = group["PERCENTAGE"].nunique()
    n_unique_geoms = group.geometry.area.nunique()

    print(f"EA_CODE: {ea_code}  |  occurrences: {n}  |  unique wards: {n_unique_wards}"
          f"  |  unique AREA values: {n_unique_areas}  |  unique PERCENTAGE values: {n_unique_pcts}"
          f"  |  unique geometry areas: {n_unique_geoms}")
    print(group[inspect_cols].to_string(index = True))
    print()

In [ ]:
# PRE-DEDUPLICATION CROSS-CHECK 3: AREA TIES.
# The deduplication strategy keeps the row with the largest AREA.
# If two rows for the same EA_CODE have identical AREA values, the
# tie-break is arbitrary (first in sort order). Flag any such ties.
area_per_dup = dup_rows.groupby("EA_CODE")["AREA"]
tied = dup_rows[
    dup_rows.duplicated(subset = ["EA_CODE", "AREA"], keep = False)
]["EA_CODE"].nunique()

print("AREA TIE CHECK")
print(f"  Duplicate EA_CODEs with tied AREA values: {tied}")
if tied > 0:
    tied_codes = dup_rows[
        dup_rows.duplicated(subset = ["EA_CODE", "AREA"], keep = False)
    ]["EA_CODE"].unique()[:5]
    print(f"  Sample tied EA_CODEs (tie-break will be arbitrary):")
    print(dup_rows[dup_rows["EA_CODE"].isin(tied_codes)][["EA_CODE", ward_col, "AREA", "PERCENTAGE"]])
print()

In [ ]:
# PRE-DEDUPLICATION CROSS-CHECK 4: GEOMETRY CONSISTENCY.
# Each row for the same EA_CODE should carry the same polygon geometry,
# since the SAL boundary does not change across ward assignments.
# Compare geometry area (shapely .area) across duplicate pairs.
# Large discrepancies indicate geometry corruption or a data join error.
dup_rows["geom_area"] = dup_rows.geometry.area
geom_spread = dup_rows.groupby("EA_CODE")["geom_area"].agg(["min", "max"])
geom_spread["rel_diff"] = (geom_spread["max"] - geom_spread["min"]) / geom_spread["max"]

# Flag EA_CODEs where geometry area differs by more than 1% across duplicates.
geom_mismatch = geom_spread[geom_spread["rel_diff"] > 0.01]

print("GEOMETRY CONSISTENCY CHECK (across duplicate pairs)")
print(f"  Duplicate groups with geometry area difference > 1%: {len(geom_mismatch)}")
if len(geom_mismatch) > 0:
    print(f"  WARNING: These EA_CODEs carry different geometries across duplicate rows:")
    print(geom_mismatch.head(10))
else:
    print(f"  All duplicate pairs carry consistent geometry.")
print()

In [ ]:
# DEDUPLICATE: sort descending by AREA, keep first (dominant ward) per EA_CODE.
sal_dedup = (
    sal_raw
    .sort_values("AREA", ascending = False)
    .drop_duplicates(subset = ["EA_CODE"], keep = "first")
    .sort_index()
    .copy()
)

print(f"AFTER DEDUPLICATION: {sal_dedup.shape[0]} features")
print(f"Unique EA_CODEs: {sal_dedup['EA_CODE'].nunique()}")
print(f"Duplicates remaining: {sal_dedup.shape[0] - sal_dedup['EA_CODE'].nunique()}")

In [ ]:
# POST-DEDUPLICATION CHECK: GEOMETRY VALIDITY AND CRS.

# Confirm final geometry is valid (no null or empty geometries).
null_geom = sal_dedup.geometry.isna().sum()
empty_geom = sal_dedup.geometry.is_empty.sum()
print()
print(f"  Null geometries : {null_geom}")
print(f"  Empty geometries: {empty_geom}")

# Confirm CRS was preserved.
print(f"  CRS: {sal_dedup.crs}")

In [ ]:
# SAVE DEDUPLICATED SHAPEFILE.
os.makedirs(OUT_SAL_DEDUP.parent, exist_ok = True)

sal_dedup.to_file(OUT_SAL_DEDUP)

print(f"SAVED: {OUT_SAL_DEDUP}")
print(f"Final shape: {sal_dedup.shape}")
